# MT05 — Vision-Language-Action: fine-tuning SmolVLA

### Lab Description

A **vision-language-action (VLA)** model maps a camera image plus a natural-language instruction directly to robot actions. **SmolVLA** is a compact (450M-parameter) open VLA from the LeRobot project, pretrained on community robot data and small enough to fine-tune on a single AMD GPU.

In MT04 you trained a tiny MLP from scratch on low-dimensional state. Here you take the **same scripted-expert demonstrations**, but record the camera image and a language instruction, store them as a **`LeRobotDataset`**, and **fine-tune the pretrained SmolVLA** to perform the robosuite reach. Comparing the zero-shot model to the fine-tuned one shows what adapting a foundation policy buys you.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium + lerobot) that this notebook is built for.

## Goals
- Record robosuite demonstrations as a `LeRobotDataset` (image + state + action + instruction)
- Load the pretrained SmolVLA policy and adapt it to a single-camera, 7-D-action robot
- Fine-tune the VLA on the AMD GPU and watch the flow-matching loss fall
- Compare the zero-shot vs. fine-tuned policy by rolling it out in robosuite

### Imports and renderer

Same EGL setup and the reliable `make_renderer` / `grab_frame` helpers from MT01. We also define the compact `state_vec` (end-effector + cube position) and the `expert_action` scripted reach reused from MT03/MT04 — this time the expert will *also* be photographed and labelled with an instruction.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import mujoco
import imageio
import torch
import robosuite as suite
from robosuite import load_composite_controller_config
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)  # hide repetitive controller-config warnings

os.makedirs("output/videos", exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

# Stable renderer for this ROCm/EGL stack (see MT01): drive mujoco.Renderer
# directly and show only the visual geom group (group 1).
def make_renderer(env, height=128, width=128):
    R = mujoco.Renderer(env.sim.model._model, height=height, width=width)
    opt = mujoco.MjvOption()
    opt.geomgroup[:] = 0
    opt.geomgroup[1] = 1
    return R, opt

def grab_frame(env, R, opt, camera="frontview"):
    mujoco.mj_forward(env.sim.model._model, env.sim.data._data)
    R.update_scene(env.sim.data._data, camera=camera, scene_option=opt)
    return R.render()

def state_vec(obs):
    # compact proprioceptive state: end-effector position (3) + cube position (3)
    return np.concatenate([obs["robot0_eef_pos"], obs["cube_pos"]]).astype(np.float32)

def expert_action(obs):
    # scripted task-space reach toward the cube (the MT03/MT04 expert)
    delta = np.clip((obs["cube_pos"] - obs["robot0_eef_pos"]) * 5, -1, 1)
    return np.array([delta[0], delta[1], delta[2], 0, 0, 0, -1.0], dtype=np.float32)

### Load the baked pretrained SmolVLA offline

The course image contains pinned snapshots of `lerobot/smolvla_base` and its SmolVLM2 backbone in its Hugging Face cache. We enable **offline** mode before loading them, so this lab never needs a runtime download and the local demonstration dataset is not looked up on the Hub.

In [ ]:
BASE = "lerobot/smolvla_base"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
POLICY_ALLOW_PATTERNS = [
    "config.json", "model.safetensors", "policy_preprocessor.json",
    "policy_preprocessor_step_5_normalizer_processor.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor_step_0_unnormalizer_processor.safetensors",
]
from huggingface_hub import snapshot_download
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# Resolve the pinned baked snapshot locally; policy loading also resolves its baked SmolVLM2 backbone.
snapshot_download(BASE, local_files_only=True, allow_patterns=POLICY_ALLOW_PATTERNS)
SmolVLAPolicy.from_pretrained(BASE, local_files_only=True)
print("SmolVLA base ready")

### Collect demonstrations as a LeRobotDataset

We run the scripted expert for several episodes (each with a freshly randomized cube) and store every step as a `LeRobotDataset` frame: the **camera image**, the **state**, the **7-D action**, and the language **task** `"lift the red cube"`. This is exactly the data format LeRobot policies train on. We keep images small (128×128) and use image (not video) storage to keep the lab fast.

In [ ]:
import shutil
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DS_ROOT = "output/mt05_dataset"
shutil.rmtree(DS_ROOT, ignore_errors=True)
FPS = 10

features = {
    "observation.state": {"dtype": "float32", "shape": (6,),
                          "names": ["eef_x", "eef_y", "eef_z", "cube_x", "cube_y", "cube_z"]},
    "observation.images.camera1": {"dtype": "image", "shape": (128, 128, 3),
                                   "names": ["height", "width", "channels"]},
    "action": {"dtype": "float32", "shape": (7,), "names": [f"a{i}" for i in range(7)]},
}

controller = load_composite_controller_config(controller="BASIC", robot="Panda")
env = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                 has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                 control_freq=20, horizon=80)
R, opt = make_renderer(env)

ds = LeRobotDataset.create("auplc/mt05_lift", fps=FPS, features=features, root=DS_ROOT,
                           use_videos=False, metadata_buffer_size=1)
for ep in range(12):
    obs = env.reset()
    for _ in range(60):
        a = expert_action(obs)
        ds.add_frame({
            "observation.state": state_vec(obs),
            "observation.images.camera1": grab_frame(env, R, opt).astype(np.uint8),
            "action": a,
            "task": "lift the red cube",
        })
        obs, reward, done, info = env.step(a)
    ds.save_episode()
ds.finalize()
R.close()
print("collected", ds.num_frames, "frames across", ds.num_episodes, "episodes")

### Load and adapt the pretrained SmolVLA

We reload the dataset with `delta_timestamps` so each sample carries a **chunk** of future actions (SmolVLA predicts an action sequence with a flow-matching head). We then build the policy from the pretrained base with `make_policy`, which loads the pretrained weights and adapts the input/output projections to our dataset — SmolVLA pads our 6-D state and 7-D action into its fixed cross-embodiment slots. Since our robosuite dataset has a single camera, we drop the base's spare camera inputs. The `preprocess`/`postprocess` pipelines handle (un)normalization and language tokenization.

In [ ]:
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.factory import make_policy, make_pre_post_processors

CHUNK = 50
delta_timestamps = {"action": [i / FPS for i in range(CHUNK)]}
ds = LeRobotDataset("auplc/mt05_lift", root=DS_ROOT, delta_timestamps=delta_timestamps)

cfg = PreTrainedConfig.from_pretrained(BASE)
cfg.pretrained_path = BASE
cfg.device = device
# our robosuite dataset has a single camera; drop the base's spare camera slots
for k in list(cfg.input_features.keys()):
    if k.startswith("observation.images.") and k != "observation.images.camera1":
        del cfg.input_features[k]

policy = make_policy(cfg, ds_meta=ds.meta)
preprocess, postprocess = make_pre_post_processors(
    policy.config, pretrained_path=BASE, dataset_stats=ds.meta.stats,
    preprocessor_overrides={"device_processor": {"device": device}})
print("SmolVLA ready:", round(sum(p.numel() for p in policy.parameters()) / 1e6, 1), "M params")

### Zero-shot baseline (before fine-tuning)

First we roll out the **un-fine-tuned** SmolVLA on the task. It has never seen this robot or camera, so its actions are essentially uninformed — the end-effector does not reliably reach the cube. We record the final distance to compare against after training.

In [ ]:
def rollout(policy, steps=60, save=None):
    policy.eval()
    policy.reset()
    env = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                     has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                     control_freq=20, horizon=steps + 5)
    R, opt = make_renderer(env)                  # 128 px: the policy's camera input (matches training)
    Rv, optv = make_renderer(env, 480, 480)      # higher-res renderer just for the playback video (480 = max offscreen fb)
    obs = env.reset()
    frames = [grab_frame(env, Rv, optv)]
    for _ in range(steps):
        rgb = grab_frame(env, R, opt)
        batch = {
            "observation.state": torch.from_numpy(state_vec(obs)).unsqueeze(0),
            "observation.images.camera1": torch.from_numpy(rgb).permute(2, 0, 1).float().unsqueeze(0) / 255.0,
            "task": ["lift the red cube"],
        }
        batch = preprocess(batch)
        with torch.no_grad():
            action = postprocess(policy.select_action(batch)).squeeze(0).cpu().numpy()[:7]
        obs, reward, done, info = env.step(action.astype(float))
        frames.append(grab_frame(env, Rv, optv))
    R.close(); Rv.close()
    dist = float(np.linalg.norm(obs["cube_pos"] - obs["robot0_eef_pos"]))
    if save:
        imageio.mimsave(save, frames, fps=20)
    return dist

zero_shot = rollout(policy, steps=60)
print(f"zero-shot (before fine-tuning) eef-to-cube distance: {zero_shot:.3f} m")

### Fine-tune the VLA on the demonstrations

We run a **deliberately short, demonstrative** fine-tune (a couple of minutes on this GPU): sample batches from the dataset, run them through the policy's `forward` (which returns the **flow-matching loss**), and step AdamW on the AMD GPU. Watch the loss fall — the model *is* learning — but this brief run is **not** enough to produce a finished policy.

> **How much training does a usable policy actually need?** Fine-tuning a 450M VLA to reliably solve even this simple reach typically takes on the order of **10,000–20,000 gradient steps** (tens of minutes to a few hours on a single GPU), plus more demonstrations. To keep the lab fast we only run a few hundred steps here; a fully-trained checkpoint is used later so you can see a polished result without waiting for a full training run.

In [ ]:
import matplotlib.pyplot as plt

loader = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4)
policy.train()

STEPS = 60  # deliberately short demo; a usable policy needs ~10k-20k steps
losses = []
it = iter(loader)
for step in range(STEPS):
    try:
        batch = next(it)
    except StopIteration:
        it = iter(loader)
        batch = next(it)
    batch = preprocess(batch)
    out = policy.forward(batch)
    loss = out[0] if isinstance(out, (tuple, list)) else out["loss"]
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().item())
    if step % 10 == 0:
        print(f"step {step:3d}  loss {losses[-1]:.4f}")
print("final fine-tuning loss:", round(losses[-1], 4))

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("step"); plt.ylabel("flow-matching loss")
plt.title("SmolVLA fine-tuning loss"); plt.yscale("log"); plt.grid(True); plt.show()

### Roll out the briefly-fine-tuned policy

Now the briefly-fine-tuned VLA drives the arm from the camera image and the instruction. After only a few hundred steps the motion already **bends toward the cube** compared to the zero-shot baseline, but it has **not** learned the full reach yet — expect an imperfect rollout. That partial progress, together with the falling loss curve, is the point: the model is on its way but undertrained. A fully-trained checkpoint gives the polished behaviour.

In [ ]:
fine_tuned = rollout(policy, steps=60, save="output/videos/mt05_smolvla.mp4")
print(f"after short fine-tune: eef-to-cube distance = {fine_tuned:.3f} m   (zero-shot was {zero_shot:.3f} m)")
print("note: still undertrained - a fully-trained checkpoint reaches ~0.04 m (cf. MT04).")

In [ ]:
Video(url="output/videos/mt05_smolvla.mp4", width=640)

### Appendix: the fully-trained checkpoint

The short fine-tune above only *starts* learning. A **fully-trained SmolVLA
checkpoint** (trained for ~1.5k steps on more demonstrations) is published on the
public Hugging Face repo
[`sonya-tw/mt05-smolvla-lift`](https://huggingface.co/sonya-tw/mt05-smolvla-lift),
so you can see the polished result without a long training run. Its pinned
snapshot is baked into the image and the cell below loads it offline, then rolls it out — it reaches the cube to
≈0.02–0.04 m, on par with the MT04 expert.

In [ ]:
import random

CKPT = "sonya-tw/mt05-smolvla-lift"                    # pinned baked HF checkpoint

# Fixed seed so this appendix rollout is reproducible
SEED = 4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    ck = SmolVLAPolicy.from_pretrained(CKPT, local_files_only=True).to(device).eval()
    ck_pre, ck_post = make_pre_post_processors(
        ck.config, pretrained_path=CKPT,
        preprocessor_overrides={"device_processor": {"device": device}})
    env_c = suite.make(env_name="Lift", robots="Panda", controller_configs=controller,
                       has_renderer=False, has_offscreen_renderer=False, use_camera_obs=False,
                       control_freq=20, horizon=80, seed=SEED)
    Rc, optc = make_renderer(env_c)                 # 128 px policy input
    Rcv, optcv = make_renderer(env_c, 480, 480)     # higher-res video (480 = max offscreen fb)
    ck.reset(); obs = env_c.reset(); frames = [grab_frame(env_c, Rcv, optcv)]
    for _ in range(60):
        rgb = grab_frame(env_c, Rc, optc)
        batch = {"observation.state": torch.from_numpy(state_vec(obs)).unsqueeze(0),
                 "observation.images.camera1": torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().unsqueeze(0) / 255.0,
                 "task": ["lift the red cube"]}
        batch = ck_pre(batch)
        with torch.no_grad():
            a = ck_post(ck.select_action(batch)).squeeze(0).cpu().numpy()[:7]
        obs, _, _, _ = env_c.step(a.astype(float)); frames.append(grab_frame(env_c, Rcv, optcv))
    Rc.close(); Rcv.close()
    dist = float(np.linalg.norm(obs["cube_pos"] - obs["robot0_eef_pos"]))
    imageio.mimsave("output/videos/mt05_checkpoint.mp4", frames, fps=20)
    print(f"fully-trained checkpoint ({CKPT}): final eef-to-cube distance = {dist:.3f} m")
    from IPython.display import display
    display(Video(url="output/videos/mt05_checkpoint.mp4", width=640))
except Exception as e:
    print("Loading the baked trained checkpoint failed:", type(e).__name__, str(e)[:160])
    print("Validate the image's offline Hugging Face cache before rerunning this appendix.")

## Conclusions

You recorded robosuite demonstrations in the `LeRobotDataset` format, loaded the pretrained **SmolVLA** vision-language-action model, adapted it to this single-camera 7-D robot, and ran a short fine-tune on the AMD GPU. The falling loss and the slightly-improved rollout show the adaptation working; the still-imperfect reach is simply because we stopped after a few hundred steps — a usable policy needs far more training (and a fully-trained checkpoint delivers the polished result).

Where MT04 trained a small MLP from scratch on low-dimensional state, MT05 *fine-tuned a pretrained foundation model* that consumes pixels and language — the exact workflow used to adapt vision-language-action models to real robots, just with more data and longer training.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT